# Codelist_Phrases — one row per phrase and term

Builds `sdtm-findings-graph/machine_actionable/Codelist_Phrases.xlsx` from the `Codelists` sheets of
the four Findings files. Every term's submission value, NCI preferred term, CDISC synonyms and NCIt
synonyms, each as a normalised phrase, with the string as published and facts about it: which field
and source it comes from, how many words it has, and how many terms and codelists share it.

Facts only. No flags, no stop list, no plural or stemming rules, no choosing: those are a consumer's
decisions. The one derived form, `phrase_compact`, is labelled as derived.

## Normalisation (stated once, for every consumer)

`phrase` = the string lower-cased, split on every character that is not a letter A–Z or a–z or a
digit 0–9, the parts joined by one space. The same rule as `Shared_Strings` in the Findings files.
A string with no letter or digit (e.g. the unit `%`) keeps its own characters, lower-cased, without
whitespace; without that it could not be found (the conformance check found it).
`phrase_compact` = `phrase` without spaces (derived).

## Inputs

| File | Sheet |
|---|---|
| `sdtm-findings-graph/machine_actionable/{Specimen,Measurement,Instrument,Immunogenicity}_Findings.xlsx` | `Codelists`, `Provenance` |
| `Reference_Versions.json` | — |

Run after `Domain_Coverage`, which checks that a codelist in several Findings files is identical.

The run **fails** when: the Findings files were built on other versions than the pin; a codelist
differs between the files; a term's submission value does not resolve to the term in the index.

## 1. Setup and inputs

In [1]:
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

BASE_DIR = Path.cwd().parent          # sdtm-findings-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/
VERSIONS = json.loads((REPO_ROOT / 'Reference_Versions.json').read_text(encoding='utf-8'))
FINDINGS_FILES = {f: BASE_DIR / 'machine_actionable' / f'{f}.xlsx'
                  for f in ['Specimen_Findings', 'Measurement_Findings', 'Instrument_Findings', 'Immunogenicity_Findings']}
OUTPUT_FILE = BASE_DIR / 'machine_actionable' / 'Codelist_Phrases.xlsx'

codelists = {}
for name, path in FINDINGS_FILES.items():
    if not path.exists():
        raise FileNotFoundError(path)
    prov = pd.read_excel(path, sheet_name='Provenance', dtype=str)
    prov = dict(zip(prov['Key'], prov['Value']))
    wrong = {k: prov[k] for k in ['SDTM_CT', 'COSMoS', 'SDTMIG', 'SDTM', 'NCIt'] if prov[k] != VERSIONS[k]}
    if wrong:
        raise RuntimeError(f'{name} was built on {wrong}, the pin is {VERSIONS}: re-run it')
    codelists[name] = pd.read_excel(path, sheet_name='Codelists', dtype=str, keep_default_na=False)
    print(f'  {path.relative_to(REPO_ROOT)}: {len(codelists[name]):,} terms')

  sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx: 3,246 terms


  sdtm-findings-graph/machine_actionable/Measurement_Findings.xlsx: 3,386 terms


  sdtm-findings-graph/machine_actionable/Instrument_Findings.xlsx: 3,273 terms


  sdtm-findings-graph/machine_actionable/Immunogenicity_Findings.xlsx: 3,679 terms


## 2. One set of codelists

A codelist in several Findings files must be identical in all of them (`referenced_by` aside, which
names the file's own columns); the run stops otherwise. The union is the scope of this file.

In [2]:
TERM_COLS = ['codelist_submission_value', 'codelist_concept_id', 'codelist_name', 'term_submission_value',
             'term_concept_id', 'term_nci_preferred_term', 'term_synonyms', 'term_ncit_synonyms']
seen, problems = {}, []
for name, df in codelists.items():
    for cl, g in df[TERM_COLS].groupby('codelist_submission_value'):
        g = g.sort_values(['term_concept_id', 'term_submission_value']).reset_index(drop=True)
        if cl in seen and not seen[cl][1].equals(g):
            problems.append(f'{cl}: {seen[cl][0]} and {name} differ')
        seen.setdefault(cl, (name, g))
if problems:
    raise RuntimeError('Codelists differ between Findings files:\n' + '\n'.join(problems))
terms = pd.concat([g for _, g in seen.values()], ignore_index=True)
in_files = {cl: sorted(n.replace('_Findings', '') for n, df in codelists.items() if cl in set(df['codelist_submission_value']))
            for cl in seen}
print(f'Codelists: {len(seen)}; terms: {len(terms):,}')

Codelists: 34; terms: 6,369


## 3. Phrases

One row per codelist, term, field and published string. `field`: submission value, NCI preferred
term, CDISC synonym (separated by `;` in SDTM CT), NCIt synonym (separated by `|`, as in the FLAT
file). `source`: CT for the first three (published in SDTM CT), NCIt for the last (NCIt FLAT file).
A string whose phrase is empty (punctuation only) is dropped and counted.

In [3]:
_TOKEN = re.compile(r'[A-Za-z0-9]+')


def norm(text):
    # Letters and digits, lower-cased, one space between. A string with none (e.g. the unit '%')
    # keeps its own characters, lower-cased, without whitespace.
    p = ' '.join(_TOKEN.findall(text.lower()))
    return p if p else ''.join(text.lower().split())


rows, empty, fallback = [], 0, []
for r in terms.itertuples(index=False):
    strings = [('submission value', 'CT', r.term_submission_value),
               ('NCI preferred term', 'CT', r.term_nci_preferred_term)]
    strings += [('CDISC synonym', 'CT', s.strip()) for s in r.term_synonyms.split(';') if s.strip()]
    strings += [('NCIt synonym', 'NCIt', s.strip()) for s in r.term_ncit_synonyms.split('|') if s.strip()]
    for field, source, text in strings:
        p = norm(text)
        if not p:
            empty += 1
            continue
        if not _TOKEN.search(text):
            fallback.append(text)
        rows.append({'phrase': p, 'phrase_compact': p.replace(' ', ''), 'as_published': text, 'field': field,
                     'source': source, 'token_count': str(len(p.split())),
                     'codelist_submission_value': r.codelist_submission_value,
                     'codelist_concept_id': r.codelist_concept_id,
                     'term_concept_id': r.term_concept_id, 'term_submission_value': r.term_submission_value})
phrases = pd.DataFrame(rows).drop_duplicates()

# Sharing, as counts (facts about the published strings).
_t = phrases.drop_duplicates(['codelist_submission_value', 'phrase', 'term_concept_id'])
_terms_in_cl = _t.groupby(['codelist_submission_value', 'phrase'])['term_concept_id'].nunique()
phrases['terms_in_codelist'] = [str(_terms_in_cl[(c, p)]) for c, p in zip(phrases['codelist_submission_value'], phrases['phrase'])]
_cls = phrases.groupby('phrase')['codelist_submission_value'].agg(lambda s: '|'.join(sorted(set(s))))
phrases['phrase_codelists'] = phrases['phrase'].map(_cls)
phrases['codelists_with_phrase'] = phrases['phrase_codelists'].map(lambda v: str(len(v.split('|'))))
_tc = phrases.drop_duplicates(['codelist_submission_value', 'phrase_compact', 'term_concept_id'])
_compact_in_cl = _tc.groupby(['codelist_submission_value', 'phrase_compact'])['term_concept_id'].nunique()
phrases['compact_terms_in_codelist'] = [str(_compact_in_cl[(c, p)]) for c, p in
                                        zip(phrases['codelist_submission_value'], phrases['phrase_compact'])]

COLS = ['phrase', 'phrase_compact', 'as_published', 'field', 'source', 'token_count',
        'codelist_submission_value', 'codelist_concept_id', 'term_concept_id', 'term_submission_value',
        'terms_in_codelist', 'compact_terms_in_codelist', 'codelists_with_phrase', 'phrase_codelists']
FIELD_ORDER = {'submission value': 0, 'NCI preferred term': 1, 'CDISC synonym': 2, 'NCIt synonym': 3}
phrases = (phrases.assign(_f=phrases['field'].map(FIELD_ORDER))
           .sort_values(['codelist_submission_value', 'phrase', 'term_concept_id', '_f', 'as_published'])
           [COLS].reset_index(drop=True))
print(f'Phrase rows: {len(phrases):,}; distinct phrases: {phrases["phrase"].nunique():,}; empty strings dropped: {empty}')
print(f'Strings without letters or digits, kept as their own characters: {sorted(set(fallback))}')

Phrase rows: 23,383; distinct phrases: 17,875; empty strings dropped: 0
Strings without letters or digits, kept as their own characters: ['%']


## 4. Conformance check

Every term's submission value, as a phrase, must find the term in the index. It does by
construction; the check guards the normalisation, not the data.

In [4]:
_keys = set(zip(phrases['codelist_submission_value'], phrases['phrase'], phrases['term_concept_id']))
bad = [(r.codelist_submission_value, r.term_submission_value) for r in terms.itertuples(index=False)
       if (r.codelist_submission_value, norm(r.term_submission_value), r.term_concept_id) not in _keys]
if bad:
    raise RuntimeError(f'Submission values that do not resolve to their term: {bad[:10]}')
missing_terms = set(terms['term_concept_id'] + '@' + terms['codelist_submission_value']) - \
    set(phrases['term_concept_id'] + '@' + phrases['codelist_submission_value'])
if missing_terms:
    raise RuntimeError(f'Terms without a phrase: {sorted(missing_terms)[:10]}')
print(f'Conformance: all {len(terms):,} terms resolve from their submission value.')

# Counts for the ReadMe.
by_cl = (phrases.drop_duplicates(['codelist_submission_value', 'phrase'])
         .assign(shared=lambda d: d['terms_in_codelist'].astype(int) > 1,
                 multi=lambda d: d['codelists_with_phrase'].astype(int) > 1)
         .groupby('codelist_submission_value').agg(phrases=('phrase', 'size'), shared=('shared', 'sum'), in_other=('multi', 'sum')))
_compact_new = (phrases.drop_duplicates(['codelist_submission_value', 'phrase_compact'])
                .query('compact_terms_in_codelist != terms_in_codelist'))
n_compact_new = len(_compact_new)
_nonascii = int(pd.concat([terms[c] for c in ['term_submission_value', 'term_nci_preferred_term', 'term_synonyms',
                                                'term_ncit_synonyms']]).str.contains(r'[^\x00-\x7F]').sum())
print(by_cl.to_string())
print(f'Compact forms that name more terms than the phrase: {n_compact_new}; strings with non-ASCII characters: {_nonascii}')

Conformance: all 6,369 terms resolve from their submission value.
                           phrases  shared  in_other
codelist_submission_value                           
AGEU                            20       0        20
CCCAT                          341       0        29
EGSTRESC                       813       0         1
EVAL                           122       0         2
FTCAT                          106       0         0
ISBDAGT                       1640       6         6
ISFTSDTL                        66       0         6
KFSS101STR                      24       0         8
KFSS102STR                      21       0         7
KFSS103STR                      21       0         7
KFSS104STR                      24       0         8
KFSS105STR                      24       0         8
KFSS106STR                      24       0         8
KFSS107STR                      21       0         7
KFSS108STR                       9       0         3
LAT                             1

## 5. Write workbook

In [5]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
GREEN = PatternFill('solid', fgColor='548235')
GREY = PatternFill('solid', fgColor='808080')
ORANGE = PatternFill('solid', fgColor='C65911')   # derived

wb = Workbook()
ws = wb.active
ws.title = 'ReadMe'
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)
f = DATA_FONT
readme_lines = [
    ('Codelist_Phrases — one row per phrase and term', title_font),
    ('', None),
    ('PURPOSE', section_font),
    ('A phrase index over the codelists the Findings files carry: every submission value, NCI preferred term,', f),
    ('  CDISC synonym and NCIt synonym of every term, as a normalised phrase, with the string as published and', f),
    ('  facts about it. So consumers that look up free text (e.g. activity labels) normalise and match the same', f),
    ('  way, and see why a hit may be weak: NCIt source, one word, shared with other terms or codelists.', f),
    ('  Facts only: no flags, no stop list, no plural or stemming rules, no choosing. Those are the consumer\'s.', f),
    ('', None),
    ('SCOPE', section_font),
    (f'The {len(seen)} codelists in the Codelists sheets of the four Findings files, complete ({len(terms):,} terms):', f),
    *[(f'  {cl}: {in_files[cl] and ", ".join(in_files[cl])}', f) for cl in sorted(seen)],
    ('Other SDTM CT codelists are not here.', f),
    ('', None),
    ('SOURCE VERSIONS', section_font),
    (f'SDTM CT {VERSIONS["SDTM_CT"]} · COSMoS {VERSIONS["COSMoS"]} · SDTMIG v{VERSIONS["SDTMIG"]} / SDTM v{VERSIONS["SDTM"]} · NCIt FLAT {VERSIONS["NCIt"]}', f),
    ('  Pinned in Reference_Versions.json (repo root); the run stops when a Findings file was built on other versions.', f),
    ('  As keys for programs: sheet Provenance.', f),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', f),
    ('Notebook: sdtm-findings-graph/notebooks/Codelist_Phrases.ipynb (run after Domain_Coverage)', f),
    ('Inputs: the Codelists and Provenance sheets of Specimen_, Measurement_, Instrument_ and Immunogenicity_Findings.xlsx', f),
    ('', None),
    ('NORMALISATION', section_font),
    ('phrase = the string lower-cased, split on every character that is not a letter A-Z or a-z or a digit 0-9,', f),
    ('  the parts joined by one space. A string with no letter or digit (e.g. the unit %) keeps its own', f),
    ('  characters, lower-cased, without whitespace. Normalise the text you look up with the same rule.', f),
    ('phrase_compact = phrase without spaces. Derived by rule, not published ("q pcr" and "qpcr" meet here).', f),
    ('', None),
    ('KNOWN GAPS', section_font),
    ('The rule drops case and punctuation; as_published keeps them. Case carries meaning in synonyms (e.g. an', f),
    ('  all-uppercase CDISC synonym is often an abbreviation), not in submission values, which are uppercase by', f),
    ('  CDISC convention. Matching case-sensitively is a consumer decision.', f),
    (f'{_nonascii} published strings contain non-ASCII characters (e.g. an accented letter); the rule splits on them.', f),
    (f'Strings with no letter or digit, kept as their own characters: {", ".join(sorted(set(fallback))) or "none"}.', f),
    ('No plural, spelling or stemming variants.', f),
    ('NCIt synonyms come from the NCI EVS FLAT file, which does not say where each synonym comes from. Some are', f),
    ('  generic or partial names; token_count and source show which ones a consumer may want to treat with care.', f),
    ('  E.g. in ISBDAGT, "HLA" is an NCIt synonym of HLA CLASS II ANTIGENS (C20705) only, not of HLA CLASS I', f),
    ('  ANTIGENS (C20706): terms_in_codelist is 1 for it.', f),
    ('codelists_with_phrase counts subset codelists too (e.g. VSRESU is a subset of UNIT).', f),
    (f'{n_compact_new} compact forms name more terms in their codelist than the phrase does (compact_terms_in_codelist).', f),
    ('', None),
    ('SHEETS', section_font),
    ('ReadMe — this sheet.', f),
    (f'Phrases — {len(phrases):,} rows, {phrases["phrase"].nunique():,} distinct phrases. One row per codelist, term, field and', f),
    ('  published string.', f),
    ('  Some phrases and terms are literally "NA" (e.g. Not Applicable in NY). Read with', f),
    ('  keep_default_na=False (pandas) or the equivalent, or they become empty values.', f),
    ('Provenance — fixed keys for programs: source versions (SDTM_CT, COSMoS, SDTMIG, SDTM, NCIt),', f),
    ('  Generated, Notebook; per data sheet Rows, Columns, Content_SHA256 (equal hash = equal data).', f),
    ('', None),
    ('COLUMNS (Phrases)', section_font),
    ('phrase, phrase_compact — see NORMALISATION.', f),
    ('as_published — the string as in SDTM CT or the NCIt FLAT file.', f),
    ('field — submission value | NCI preferred term | CDISC synonym | NCIt synonym.', f),
    ('source — CT (published in SDTM CT: the first three fields) | NCIt (NCIt FLAT file).', f),
    ('token_count — words in phrase.', f),
    ('codelist_submission_value, codelist_concept_id, term_concept_id, term_submission_value — the term.', f),
    ('terms_in_codelist — terms in this codelist that have this phrase. More than 1: the phrase identifies none of them.', f),
    ('compact_terms_in_codelist — the same for phrase_compact.', f),
    ('codelists_with_phrase, phrase_codelists — the codelists in this file that have this phrase (count; list, |).', f),
    ('', None),
    ('PHRASES PER CODELIST (distinct phrases / named by more than one term / also in another codelist)', section_font),
    *[(f'  {cl}: {int(r.phrases):,} / {int(r.shared):,} / {int(r.in_other):,}', f) for cl, r in by_cl.iterrows()],
    ('', None),
    ('CHECKS (the run fails otherwise)', section_font),
    ('The Findings files were built on the pinned versions.', f),
    ('A codelist in several Findings files is identical in all of them.', f),
    ('Every term\'s submission value, as a phrase, resolves to the term.', f),
    ('', None),
    ('Not an official CDISC product.', f),
]
for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font
ws.column_dimensions['A'].width = 110

ws_p = wb.create_sheet('Phrases')
FILLS = {c: GREEN for c in COLS}
FILLS['phrase_compact'] = ORANGE
for c in ['token_count', 'terms_in_codelist', 'compact_terms_in_codelist', 'codelists_with_phrase', 'phrase_codelists']:
    FILLS[c] = GREY
WIDTHS = {'phrase': 40, 'phrase_compact': 36, 'as_published': 44, 'field': 18, 'source': 8, 'token_count': 8,
          'codelist_submission_value': 14, 'codelist_concept_id': 12, 'term_concept_id': 12,
          'term_submission_value': 40, 'terms_in_codelist': 10, 'compact_terms_in_codelist': 10,
          'codelists_with_phrase': 10, 'phrase_codelists': 24}
for ci, name in enumerate(COLS, 1):
    cell = ws_p.cell(row=1, column=ci, value=name)
    cell.font, cell.fill = HEADER_FONT, FILLS[name]
    cell.alignment = Alignment(wrap_text=True, vertical='top')
    ws_p.column_dimensions[get_column_letter(ci)].width = WIDTHS[name]
for ri, row in enumerate(phrases.itertuples(index=False, name=None), 2):
    for ci, val in enumerate(row, 1):
        ws_p.cell(row=ri, column=ci, value=val if val != '' else None).font = DATA_FONT
ws_p.freeze_panes = 'A2'
ws_p.auto_filter.ref = f'A1:{get_column_letter(len(COLS))}1'
wb.save(OUTPUT_FILE)
print(f'Written: {OUTPUT_FILE.relative_to(REPO_ROOT)}')

Written: sdtm-findings-graph/machine_actionable/Codelist_Phrases.xlsx


In [6]:
# ── Provenance sheet ──
# Fixed keys for programs: the pinned source versions, and per data sheet its row count, column
# names and a content hash. The hash is computed from the file as saved, read back as text, so a
# consumer can recompute it; the xlsx file hash changes on every save, the content hash only when
# the data change.
import hashlib
from openpyxl import load_workbook as _load_workbook
from openpyxl.styles import Font as _Font, PatternFill as _PatternFill

PROVENANCE_VERSION_KEYS = ['SDTM_CT', 'COSMoS', 'SDTMIG', 'SDTM', 'NCIt']
PROVENANCE_METHOD = ('SHA-256 (UTF-8) of the sheet read as text, empty cells as "" (pandas read_excel, dtype=str): '
                     'header and rows, cells joined by tab, rows by newline. Equal hash = equal data.')


def sheet_sha256(df):
    lines = ['\t'.join(df.columns)] + ['\t'.join(row) for row in df.itertuples(index=False, name=None)]
    return hashlib.sha256('\n'.join(lines).encode('utf-8')).hexdigest()


def add_provenance(path, notebook):
    """Add a Provenance sheet (Key, Value) to a saved workbook."""
    sheets = pd.read_excel(path, sheet_name=None, dtype=str)
    if 'Provenance' in sheets:
        raise RuntimeError(f'{path.name} already has a Provenance sheet')
    rows = [(k, VERSIONS[k]) for k in PROVENANCE_VERSION_KEYS]
    rows += [('Generated', f'{datetime.now():%Y-%m-%d %H:%M}'), ('Notebook', notebook)]
    for name, df in sheets.items():
        if name == 'ReadMe':
            continue
        df = df.fillna('')
        rows += [(f'{name}.Rows', str(len(df))), (f'{name}.Columns', '|'.join(df.columns)),
                 (f'{name}.Content_SHA256', sheet_sha256(df))]
    rows.append(('Content_SHA256.Method', PROVENANCE_METHOD))
    wb_p = _load_workbook(path)
    ws_p = wb_p.create_sheet('Provenance')
    for ci, head in enumerate(['Key', 'Value'], 1):
        cell = ws_p.cell(row=1, column=ci, value=head)
        cell.font = _Font(name='Arial', bold=True, size=10, color='FFFFFF')
        cell.fill = _PatternFill('solid', fgColor='808080')
    for ri, (k, v) in enumerate(rows, 2):
        ws_p.cell(row=ri, column=1, value=k).font = _Font(name='Arial', size=10)
        ws_p.cell(row=ri, column=2, value=v).font = _Font(name='Arial', size=10)
    ws_p.column_dimensions['A'].width = 34
    ws_p.column_dimensions['B'].width = 90
    ws_p.freeze_panes = 'A2'
    wb_p.save(path)
    print(f'Provenance: {path.name}, {len(rows)} keys')


add_provenance(OUTPUT_FILE, 'sdtm-findings-graph/notebooks/Codelist_Phrases.ipynb')

Provenance: Codelist_Phrases.xlsx, 11 keys
